# Coronal Mass Ejections — Implementation / 구현

**Paper**: Chen, P. F. (2011). *Coronal Mass Ejections: Models and Their Observational Basis*. Living Reviews in Solar Physics, 8, 1. DOI: 10.12942/lrsp-2011-1

이 노트북은 Chen (2011) 리뷰의 **핵심 CME 모델과 운동학**을 선택적으로 구현합니다. 리뷰이므로 전체 시뮬레이션은 불가능하고, **트리거 조건 + 운동학 + 관측 서명**의 최소 모델을 NumPy로 구현합니다.

This notebook selectively implements **key CME models and kinematics** from Chen (2011). Since it is a review, we build minimal NumPy versions of the trigger criteria, the kinematics, and the observational signatures.

**Contents / 구성:**
1. Potential field extrapolation & decay index / 포텐셜 장과 decay index
2. Torus instability criterion $n>1.5$ / 토러스 불안정 조건
3. Kink instability threshold / 킹크 불안정 임계값
4. Catastrophe model: loss of equilibrium / 격변 모델
5. CME three-phase kinematics (height-time) / CME 3단계 운동학
6. Interplanetary drag & arrival time / 행성간 drag와 도달 시간
7. Synthetic magnetic cloud signature / 합성 magnetic cloud

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

R_SUN = 6.96e8
AU = 1.496e11

## Part 1: Potential Field and Decay Index / 포텐셜 장과 decay index

A simple **bipole** potential field above the photosphere. For two magnetic charges $\pm q$ at depths $\pm d$, the vertical field at height $h$ on the polarity inversion line (PIL) is:

$$
B_z(h) = \frac{2q\,d}{(h^2+d^2)^{3/2}}
$$

and the **horizontal (external-guiding) field** on the PIL:

$$
B_{\rm ext}(h) \propto \frac{1}{(h^2 + d^2)^{3/2}}\cdot\text{(lateral factor)}
$$

**Decay index**: $n(h) = -\partial \ln B_{\rm ext}/\partial\ln h$.

간단한 양극자 장의 PIL 위 수평 자기장에서 decay index를 계산하면 $n>1.5$ (토러스 불안정)가 되는 높이가 드러남.

In [ ]:
def bipole_horizontal_field(h: np.ndarray, d: float = 0.05) -> np.ndarray:
    """Return the horizontal guide field on the PIL for a subphotospheric bipole.

    Args:
        h: Height above the photosphere, in units of the footpoint half-separation.
        d: Subphotospheric charge depth (same units).

    Returns:
        Horizontal field strength B_ext(h), arbitrary units.
    """
    return 1.0 / (h**2 + d**2) ** 1.5


def decay_index(h: np.ndarray, B_func) -> np.ndarray:
    """Compute n(h) = -d ln B / d ln h numerically."""
    B = B_func(h)
    dlnB = np.gradient(np.log(B), np.log(h))
    return -dlnB


h = np.logspace(-2, 1.5, 400)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for d in [0.01, 0.05, 0.2]:
    B_ext = bipole_horizontal_field(h, d=d)
    axes[0].loglog(h, B_ext, label=f'depth d={d}')
axes[0].set_xlabel('height h (half-separation units)')
axes[0].set_ylabel('B_ext(h)')
axes[0].set_title('External guide field decays with height')
axes[0].legend()
axes[0].grid(alpha=0.3, which='both')

for d in [0.01, 0.05, 0.2]:
    n = decay_index(h, lambda hh: bipole_horizontal_field(hh, d=d))
    axes[1].semilogx(h, n, label=f'd={d}')
axes[1].axhline(1.5, color='r', ls='--', label='torus-unstable threshold n=1.5')
axes[1].set_xlabel('height h')
axes[1].set_ylabel('decay index n(h)')
axes[1].set_title('Decay index crosses 1.5 at the critical height')
axes[1].legend()
axes[1].grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

for d in [0.01, 0.05, 0.2]:
    n = decay_index(h, lambda hh: bipole_horizontal_field(hh, d=d))
    idx = np.argmin(np.abs(n - 1.5))
    print(f'd = {d}: torus-unstable above h ≈ {h[idx]:.2f}')

## Part 2: Torus Instability Diagnosis for a Synthetic Active Region / 가상 AR의 토러스 불안정 진단

We mock up a more realistic AR: two bipolar groups with non-trivial geometry. For each candidate launch height, we compute $n(h)$ along the PIL. This is the operational workflow behind modern ML CME predictions (SHARP parameters → $n>1.5$ boundary).

**현대 ML CME 예측 워크플로**를 흉내: AR의 벡터 자기장에서 PIL 위 $n(h)$를 계산 → 임계 높이 평가 → 분출 가능성 산출.

In [ ]:
def point_charge_field(x, y, z, charges):
    """Vector magnetic field of a set of subphotospheric point charges.

    Args:
        x, y, z: Evaluation coordinates (arrays of same shape).
        charges: List of (q, x0, y0, z0) — charge sign/strength and location.

    Returns:
        Tuple (Bx, By, Bz).
    """
    Bx = np.zeros_like(x)
    By = np.zeros_like(x)
    Bz = np.zeros_like(x)
    for q, x0, y0, z0 in charges:
        dx = x - x0
        dy = y - y0
        dz = z - z0
        r = np.sqrt(dx**2 + dy**2 + dz**2) + 1e-9
        Bx += q * dx / r**3
        By += q * dy / r**3
        Bz += q * dz / r**3
    return Bx, By, Bz


charges = [
    (+1.0, -0.5, 0.0, -0.1),
    (-1.0, +0.5, 0.0, -0.1),
    (+0.3, -0.5, 1.0, -0.05),
    (-0.3, +0.5, 1.0, -0.05),
]

h = np.linspace(0.01, 2.0, 200)
x = np.zeros_like(h)
y = np.zeros_like(h)
Bx, By, Bz = point_charge_field(x, y, h, charges)
B_horiz = np.sqrt(Bx**2 + By**2)
n = -np.gradient(np.log(B_horiz), np.log(h))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(h, B_horiz)
axes[0].set_xlabel('height h')
axes[0].set_ylabel('|B_horiz|(h) on PIL')
axes[0].set_yscale('log')
axes[0].set_title('Horizontal field on the PIL — synthetic AR')
axes[0].grid(alpha=0.3, which='both')

axes[1].plot(h, n)
axes[1].axhline(1.5, color='r', ls='--', label='torus threshold')
axes[1].set_xlabel('height h')
axes[1].set_ylabel('n(h)')
axes[1].set_title('Decay index on PIL — eruption prediction')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

h_crit = h[np.argmin(np.abs(n - 1.5))]
print(f'Torus-unstable height h_crit ≈ {h_crit:.2f}')

## Part 3: Kink Instability Threshold / 킹크 불안정 임계값

Linear Kruskal-Shafranov threshold for a straight cylindrical flux tube: instability sets in at twist $\Phi = 2\pi$. For coronal arched flux ropes, numerical work (Török & Kliem 2005) gives $\Phi_{\rm crit}\approx 2.5\pi$. We visualize the **growth rate vs. twist**.

$$\text{growth rate} \propto \sqrt{\max(0,\,\Phi - \Phi_{\rm crit})}\cdot v_A / R_0$$

twist이 임계값을 넘으면 성장률이 빠르게 증가 — 가상 kink 성장 모델.

In [ ]:
def kink_growth_rate(twist: np.ndarray, twist_crit: float = 2.5 * np.pi,
                     v_A: float = 1e6, R0: float = 1e7) -> np.ndarray:
    """Heuristic linear growth rate σ(Φ) of a kinked arched flux rope.

    Args:
        twist: Total twist Φ (radians).
        twist_crit: Critical twist (Kruskal-Shafranov: 2π; arched: ~2.5π).
        v_A: Alfvén speed in the rope (m/s).
        R0: Rope major radius (m).

    Returns:
        Growth rate (1/s), zero below threshold.
    """
    excess = np.clip(twist - twist_crit, 0, None)
    return np.sqrt(excess) * v_A / R0


twist = np.linspace(0, 5 * np.pi, 400)
for phi_c, label in [(2 * np.pi, 'straight (KS)'), (2.5 * np.pi, 'arched (TK05)')]:
    sigma = kink_growth_rate(twist, twist_crit=phi_c)
    plt.plot(twist / np.pi, sigma, label=f'$\\Phi_c$ = {label}')
plt.xlabel(r'total twist $\Phi / \pi$')
plt.ylabel('growth rate (1/s)')
plt.title('Kink instability: threshold and growth rate')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: Catastrophe Model — Loss of Equilibrium / 격변 모델

A schematic Forbes-type equilibrium: flux rope height $h$ equilibrates between a destabilizing magnetic force (proportional to rope current) and a stabilizing background field. As photospheric convergence reduces the separation parameter $\lambda$, the equilibrium curve $h(\lambda)$ develops a **fold point** — beyond which no equilibrium exists.

$$
0 = f_{\rm destab}(h) - f_{\rm stab}(h, \lambda)
$$

대류 수렴에 의해 $\lambda$가 감소하면서 평형 곡선이 fold를 지나면 격변이 발생.

In [ ]:
def catastrophe_equilibrium(lam: float) -> np.ndarray:
    """Return all equilibrium heights h satisfying a schematic Forbes equation.

    The toy equation used: 1/h - λ*h = 0.5, reshaped so that λ_c ≈ 0.125 gives a
    fold point (physically: a parametric model of rope height vs. convergence).

    Args:
        lam: Convergence parameter (larger → stronger photospheric convergence).

    Returns:
        Array of equilibrium heights (could be 0, 1, or 2 real roots).
    """
    coeffs = [lam, -0.5, -1.0, 0.0]
    roots = np.roots(coeffs)
    return np.sort(roots[np.abs(roots.imag) < 1e-8].real)


lams = np.linspace(0.01, 0.2, 200)
upper = []
lower = []
for lam in lams:
    rs = catastrophe_equilibrium(lam)
    rs = rs[rs > 0]
    if len(rs) >= 2:
        lower.append((lam, rs[0]))
        upper.append((lam, rs[-1]))

lower = np.array(lower)
upper = np.array(upper)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(lower[:, 0], lower[:, 1], 'g-', label='stable branch')
ax.plot(upper[:, 0], upper[:, 1], 'r--', label='unstable branch')
fold_lam = lower[-1, 0]
fold_h = lower[-1, 1]
ax.scatter([fold_lam], [fold_h], color='black', zorder=5, s=80, label=f'fold at λ≈{fold_lam:.3f}')
ax.annotate('catastrophe →', xy=(fold_lam, fold_h), xytext=(fold_lam - 0.05, fold_h + 1.5),
            arrowprops=dict(arrowstyle='->'))
ax.set_xlabel('convergence parameter λ')
ax.set_ylabel('equilibrium rope height h')
ax.set_title('Catastrophe model: loss of equilibrium at the fold point')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 5: CME Three-Phase Kinematics / CME 3단계 운동학

Reproduce the classic Chen (2011) Fig. 22 schematic: (i) slow rise, (ii) impulsive acceleration, (iii) propagation. Height-time curve observed in LASCO/SDO data.

LASCO/SDO에서 관측되는 고전적 h-t 곡선 재현: 느린 상승 → 충격적 가속 → 정상 전파.

In [ ]:
def cme_three_phase_height(t: np.ndarray, t0: float = 0.0, t1: float = 20.0,
                           t2: float = 40.0, v0: float = 1e3, a_imp: float = 5e2,
                           v_cruise: float = 1.2e3) -> np.ndarray:
    """Analytic three-phase CME height-time curve.

    Args:
        t: Time array (min).
        t0, t1, t2: Phase boundaries (min).
        v0: Slow-rise speed (km/s).
        a_imp: Impulsive acceleration (m/s^2).
        v_cruise: Cruise speed (km/s).

    Returns:
        Height in R_sun.
    """
    sec = 60.0
    h = np.zeros_like(t)
    v0_m = v0 * 1e3
    v_cr_m = v_cruise * 1e3
    phase1 = t < t1
    phase2 = (t >= t1) & (t < t2)
    phase3 = t >= t2
    h1_end = v0_m * (t1 - t0) * sec
    h2_end = h1_end + v0_m * (t2 - t1) * sec + 0.5 * a_imp * ((t2 - t1) * sec) ** 2
    h[phase1] = v0_m * (t[phase1] - t0) * sec
    h[phase2] = h1_end + v0_m * (t[phase2] - t1) * sec + 0.5 * a_imp * ((t[phase2] - t1) * sec) ** 2
    h[phase3] = h2_end + v_cr_m * (t[phase3] - t2) * sec
    return h / R_SUN


t = np.linspace(0, 120, 600)
h = cme_three_phase_height(t)
v = np.gradient(h * R_SUN, t * 60) / 1e3
a = np.gradient(v * 1e3, t * 60)

fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
for ax, y, label, color in zip(
    axes, [h, v, a], ['height (R_sun)', 'speed (km/s)', 'accel (m/s^2)'],
    ['C0', 'C1', 'C2']
):
    ax.plot(t, y, color=color)
    ax.axvline(20, color='k', ls=':', alpha=0.5)
    ax.axvline(40, color='k', ls=':', alpha=0.5)
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
axes[0].text(10, 1.5, 'slow rise', ha='center')
axes[0].text(30, 3.5, 'impulsive', ha='center')
axes[0].text(80, 10, 'propagation', ha='center')
axes[2].set_xlabel('time (min)')
plt.suptitle('Three-phase CME kinematics (Chen 2011 Fig. 22)')
plt.tight_layout()
plt.show()

## Part 6: Interplanetary Drag & Arrival Time / 행성간 drag와 도달 시간

Integrate

$$
\frac{du}{dt} = -\gamma\,(u - u_{\rm SW})\,|u - u_{\rm SW}|
$$

from 20 $R_\odot$ to 1 AU for different launch speeds. This is the core of ENLIL/EUHFORIA operational forecasting.

여러 launch 속도에 대해 drag 방정식을 적분하여 1 AU 도달 시간·속도 예측.

In [ ]:
def cme_propagation(u0: float, u_sw: float = 400e3, gamma: float = 2e-8,
                    r0: float = 20 * R_SUN, r_end: float = AU) -> tuple:
    """Integrate the ICME drag-based propagation from r0 to r_end.

    Args:
        u0: Launch speed at r0 (m/s).
        u_sw: Ambient solar wind speed (m/s).
        gamma: Drag parameter (1/m). Tuned to match observed ~2–3 day transit.
        r0: Inner boundary (m).
        r_end: Outer boundary (m).

    Returns:
        Tuple of arrays (t in hours, r in AU, u in km/s).
    """
    def rhs(t, y):
        r, u = y
        du = -gamma * (u - u_sw) * abs(u - u_sw)
        return [u, du]

    def hit(t, y):
        return y[0] - r_end

    hit.terminal = True
    sol = solve_ivp(rhs, (0, 1e7), [r0, u0], events=hit, max_step=600, rtol=1e-6)
    return sol.t / 3600, sol.y[0] / AU, sol.y[1] / 1e3


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for u0_km in [400, 800, 1200, 1800, 2400]:
    t_hr, r_au, u_km = cme_propagation(u0_km * 1e3)
    axes[0].plot(t_hr, r_au, label=f'u₀={u0_km} km/s')
    axes[1].plot(t_hr, u_km, label=f'u₀={u0_km}')
axes[0].axhline(1, color='k', ls=':', alpha=0.4)
axes[0].set_xlabel('time (hr)')
axes[0].set_ylabel('r (AU)')
axes[0].set_title('CME trajectory to 1 AU')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].axhline(400, color='k', ls=':', alpha=0.4, label='ambient SW')
axes[1].set_xlabel('time (hr)')
axes[1].set_ylabel('u (km/s)')
axes[1].set_title('Speed evolution: fast decelerate, slow accelerate')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Arrival times at 1 AU:')
for u0_km in [400, 800, 1200, 1800, 2400]:
    t_hr, r_au, u_km = cme_propagation(u0_km * 1e3)
    print(f'  u₀ = {u0_km:4d} km/s → {t_hr[-1]:.1f} h ({t_hr[-1]/24:.2f} d), arrival speed {u_km[-1]:.0f} km/s')

## Part 7: Synthetic Magnetic Cloud Signature / 합성 magnetic cloud

A magnetic cloud is a **force-free flux rope** passing over the spacecraft. Lundquist's (1950) cylindrically symmetric solution:

$$
B_z(r) = B_0\,J_0(\alpha r), \quad B_\phi(r) = B_0\,H\,J_1(\alpha r)
$$

where $H=\pm 1$ is the chirality. As the spacecraft traverses the rope along a chord, it sees a **smooth rotation** of $\mathbf{B}$ — the hallmark in-situ signature.

magnetic cloud의 특징인 **부드러운 자기장 회전** 재현. Lundquist force-free 해를 chord를 따라 sampling.

In [ ]:
from scipy.special import j0, j1


def lundquist_field(r: np.ndarray, phi: np.ndarray, B0: float = 20.0,
                    alpha: float = 2.405, H: int = 1) -> tuple:
    """Lundquist force-free cylindrical solution in (r, phi) local to the rope axis.

    Args:
        r: Radial distance from rope axis (normalized so α r = first zero at r=1).
        phi: Azimuth angle inside the rope.
        B0: Axial field amplitude (nT).
        alpha: Force-free parameter (2.405 → first zero of J0 at r=1).
        H: Chirality (+1 or -1).

    Returns:
        Tuple (B_axial, B_azimuth) components inside the rope.
    """
    B_axial = B0 * j0(alpha * r)
    B_azim = B0 * H * j1(alpha * r)
    return B_axial, B_azim


n = 400
y_chord = 0.3
x_chord = np.linspace(-1.0, 1.0, n)
r_chord = np.sqrt(x_chord**2 + y_chord**2)
phi_chord = np.arctan2(y_chord, x_chord)

B_ax, B_az = lundquist_field(r_chord, phi_chord)
Bx = B_ax
By = -B_az * np.sin(phi_chord)
Bz = B_az * np.cos(phi_chord)
Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)
theta_B = np.degrees(np.arctan2(By, Bx))

t_hr = np.linspace(0, 24, n)

fig, axes = plt.subplots(4, 1, figsize=(11, 11), sharex=True)
axes[0].plot(t_hr, Bmag, lw=2)
axes[0].set_ylabel('|B| (nT)')
axes[0].set_title('Synthetic magnetic cloud traversal (Lundquist flux rope)')
axes[0].grid(alpha=0.3)

axes[1].plot(t_hr, Bx, label='Bx (axial)')
axes[1].plot(t_hr, By, label='By')
axes[1].plot(t_hr, Bz, label='Bz')
axes[1].set_ylabel('B (nT)')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(t_hr, theta_B)
axes[2].set_ylabel(r'angle $\theta_B$ (deg)')
axes[2].set_title('Smooth rotation of B — flux rope signature')
axes[2].grid(alpha=0.3)

T_expected = 1e5
T_obs = T_expected * (0.3 + 0.3 * np.exp(-((t_hr - 12) / 5) ** 2))
axes[3].plot(t_hr, T_obs / T_expected)
axes[3].axhline(0.5, color='r', ls='--', label=r'T/T_exp = 0.5')
axes[3].set_ylabel(r'$T_p/T_{\rm expected}$')
axes[3].set_xlabel('time in cloud (hr)')
axes[3].set_title('Low proton temperature — another magnetic-cloud signature')
axes[3].legend()
axes[3].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Summary / 요약

| Concept / 개념 | In Chen 2011 | Our implementation | Modern tools / 현대 도구 |
|---|---|---|---|
| Potential-field extrapolation | §3 | Part 1: analytic dipole | PFSS, SHARP |
| Decay index / torus instability | §4.4 | Part 2: synthetic AR | SHARP parameters, ML |
| Kink instability threshold | §4.4 | Part 3: growth-rate curve | Török-Kliem 3D MHD |
| Catastrophe loss of equilibrium | §4.1 | Part 4: fold-point diagram | Forbes-Isenberg models |
| Three-phase kinematics | §5 | Part 5: h-t curve | GCS fitting, AIA tracking |
| Drag-based propagation | §5 | Part 6: ODE integration | ENLIL, EUHFORIA, DBM |
| Magnetic cloud signature | §5 | Part 7: Lundquist flux rope | in-situ MC fitting (L97) |

**Key lessons / 핵심 교훈:**

1. **$n>1.5$ is computable from observed photospheric magnetograms.** It is the single best quantitative eruption predictor. / $n>1.5$는 magnetogram에서 직접 계산 가능한 최강 정량 지표.
2. **The kink threshold $\approx 2.5\pi$ is gentle** — real ARs often reach $\sim 3\pi$ before erupting. / kink 임계값은 완만하나 실제 AR은 $\sim 3\pi$에서 분출.
3. **Catastrophe is a saddle-node bifurcation**, hence the smooth slow-rise precursor. / 격변은 saddle-node 분기 — 느린 상승 전조와 일치.
4. **Three phases are visible even in simple kinematics.** The impulsive phase is where models must deliver. / 세 단계는 단순 운동학에서도 분명 — 충격 단계가 모델의 핵심 시험대.
5. **Drag homogenizes speeds.** 1 AU CME speeds cluster in 400–800 km/s regardless of launch speed. / drag가 속도를 균일화 — 1 AU에서 400–800 km/s 집중.
6. **Lundquist rope gives the canonical magnetic-cloud shape** — smooth rotation is the telltale. / Lundquist 로프가 표준 MC 모양 — 부드러운 회전이 결정적 표식.

**Connections to next papers / 다음 논문과의 연결:**

- **LRSP #9 Priest (2014)** — provides the force-free field theory this review stands on.
- **LRSP #21 Ofman (2010)** — describes the solar wind into which CMEs decelerate.
- **Forbes & Priest, Antiochos, Moore, Török-Kliem** — original sources for the four trigger models.